### Import

In [15]:
import re
import pickle 
import numpy as np
import pandas as pd
import datetime as dt

pd.set_option('display.max_columns', 500)

### General parameters

In [16]:
mimiciii = "PATH TO DATA/mimic-iii(1.4)/"
output  = "../Data/csvExtract/"

### Read cohort hospital admission ids

In [3]:
cohort_subject_id_icu = []

with open("../Data/Cohort/cohort1_subject_id.txt", "r") as f:
    for subject_id in f:
        cohort_subject_id_icu.append(int(subject_id.strip()))
        
len(cohort_subject_id_icu)

15100

In [4]:
cohort_hadm_id_icu = []

with open("../Data/Cohort/cohort1_hadm_id.txt", "r") as f:
    for hadm_id in f:
        cohort_hadm_id_icu.append(int(hadm_id.strip()))
        
len(cohort_hadm_id_icu)

18070

In [5]:
cohort_stay_id_icu = []

with open("../Data/Cohort/cohort1_stay_id.txt", "r") as f:
    for stay_id in f:
        cohort_stay_id_icu.append(int(stay_id.strip()))
        
len(cohort_stay_id_icu)

19351

### patients

In [6]:
patients = pd.read_csv(mimiciii + "PATIENTS.csv")

In [7]:
patients['DOD'] = pd.to_datetime(patients['DOD'])
patients.drop(columns=['ROW_ID', 'DOD_HOSP', 'DOD_SSN'], inplace=True)
patients = patients[patients.SUBJECT_ID.isin(cohort_subject_id_icu)]

In [8]:
gender_map = {'M': 1, 'F': 2}

def transform_gender(gender_series):
    return {'GENDER':gender_series.fillna('').apply(lambda s: gender_map[s] if s in gender_map else gender_map[''])}

patients.update(transform_gender(patients.GENDER))

In [9]:
patients.head(3)

In [10]:
print(patients.SUBJECT_ID.nunique())
print(patients[patients.GENDER == 2].shape[0])
print(patients[patients.DOD.notnull()].shape[0])

15100
6614
4384


### admission

In [3]:
admissions = pd.read_csv(mimiciii + "ADMISSIONS.csv")

In [4]:
admissions = admissions[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ETHNICITY', 
                         'DIAGNOSIS', 'HOSPITAL_EXPIRE_FLAG']]

In [5]:
admissions['ADMITTIME'] = pd.to_datetime(admissions['ADMITTIME'])
admissions['DISCHTIME'] = pd.to_datetime(admissions['DISCHTIME'])
admissions['DEATHTIME'] = pd.to_datetime(admissions['DEATHTIME'])

In [6]:
def transform_race_into_id(df):
    
    df.ETHNICITY.fillna('nodx', inplace=True)
    dx_type = df.ETHNICITY.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['ETHNICITY'] = df['ETHNICITY'].map(dictionary)
    
    return df, dictionary

In [7]:
admissions, race_dictionary = transform_race_into_id(admissions)

In [16]:
admissions = admissions[admissions.HADM_ID.isin(cohort_hadm_id_icu)]

admissions = admissions[['SUBJECT_ID', 'HADM_ID', 'ETHNICITY', 'DIAGNOSIS', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME',
                         'HOSPITAL_EXPIRE_FLAG']]

In [17]:
admissions.head(3)

In [18]:
print(admissions.SUBJECT_ID.nunique())
print(admissions.HADM_ID.nunique())
print(admissions[admissions.HOSPITAL_EXPIRE_FLAG == 1].shape[0])

15100
18070
1907


### merge admission and patients

In [19]:
cohort_df = pd.merge(admissions, patients, on=['SUBJECT_ID'], how='left')

In [20]:
cohort_df['DEATH_TIME_DISCH'] = cohort_df['DOD'] - cohort_df['DISCHTIME']
cohort_df['DEATH_TIME_DISCH'] = pd.to_timedelta(cohort_df['DEATH_TIME_DISCH'])
cohort_df['DEATH_TIME_DISCH'] = (cohort_df.DEATH_TIME_DISCH / pd.Timedelta(days = 1)).round(0)
cohort_df['DEATH_TIME_DISCH'] = cohort_df['DEATH_TIME_DISCH'] + 1

In [21]:
cohort_df['DOB'] = pd.to_datetime(cohort_df['DOB'], format= "%Y-%m-%d")
cohort_df['DOB'] = cohort_df.DOB.dt.year
cohort_df['ADMITTYEAR'] = pd.to_datetime(cohort_df['ADMITTIME'], format= "%Y-%m-%d")
cohort_df['ADMITTYEAR'] = cohort_df.ADMITTYEAR.dt.year
cohort_df['AGE'] = cohort_df['ADMITTYEAR'] - cohort_df['DOB']
cohort_df.drop(columns=['ADMITTYEAR', 'DOB'], inplace=True)

In [22]:
cohort_df = cohort_df[['SUBJECT_ID', 'HADM_ID', 'GENDER', 'AGE', 'ETHNICITY', 'DIAGNOSIS', 'ADMITTIME', 
                       'DISCHTIME', 'DEATHTIME', 'DOD', 'HOSPITAL_EXPIRE_FLAG', 'DEATH_TIME_DISCH', 'EXPIRE_FLAG']]

In [23]:
cohort_df.head(3)

In [24]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df[cohort_df.HOSPITAL_EXPIRE_FLAG == 1].shape[0])
print(cohort_df[cohort_df.DEATH_TIME_DISCH.notnull()].shape[0])

15100
18070
1907
5631


### icustays

In [25]:
icustays = pd.read_csv(mimiciii + "ICUSTAYS.csv")

In [26]:
icustays.drop(columns=['ROW_ID', 'DBSOURCE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 
                       'FIRST_WARDID', 'LAST_WARDID'], inplace=True)

In [27]:
icustays['INTIME']  = pd.to_datetime(icustays['INTIME'])
icustays['OUTTIME'] = pd.to_datetime(icustays['OUTTIME'])
icustays['LOS'] = icustays['LOS'].round(1)

In [28]:
icustays = icustays[icustays.ICUSTAY_ID.isin(cohort_stay_id_icu)]

In [29]:
icustays.head(3)

In [30]:
print(icustays.SUBJECT_ID.nunique())
print(icustays.HADM_ID.nunique())
print(icustays.ICUSTAY_ID.nunique())

15100
18070
19351


### merge cohort and icu

In [31]:
cohort_df = pd.merge(cohort_df, icustays, on=['SUBJECT_ID', 'HADM_ID'], how='left')

In [32]:
cohort_df['ADM_IN_DIFF'] = cohort_df['INTIME'] - cohort_df['ADMITTIME']
cohort_df['ADM_IN_DIFF'] = pd.to_timedelta(cohort_df['ADM_IN_DIFF'])
cohort_df['ADM_IN_DIFF'] = (cohort_df.ADM_IN_DIFF / pd.Timedelta(hours = 1)).round(1)

cohort_df['OUT_DIS_DIFF'] = cohort_df['DISCHTIME'] - cohort_df['OUTTIME']
cohort_df['OUT_DIS_DIFF'] = pd.to_timedelta(cohort_df['OUT_DIS_DIFF'])
cohort_df['OUT_DIS_DIFF'] = (cohort_df.OUT_DIS_DIFF / pd.Timedelta(hours = 1)).round(1)

In [33]:
cohort_df = cohort_df[(cohort_df.ADM_IN_DIFF > -24)]
cohort_df = cohort_df[~(((cohort_df.OUT_DIS_DIFF < -24) & (cohort_df.HOSPITAL_EXPIRE_FLAG == 0)))]

condition_1 = (cohort_df.ADM_IN_DIFF < 0)
cohort_df.loc[condition_1, 'ADMITTIME']  = cohort_df.loc[condition_1, 'INTIME']

condition_2 = ((cohort_df.OUT_DIS_DIFF < 0) & (cohort_df.HOSPITAL_EXPIRE_FLAG == 0))
cohort_df.loc[condition_2, 'DISCHTIME']  = cohort_df.loc[condition_2, 'OUTTIME']

cohort_df = cohort_df[cohort_df.OUT_DIS_DIFF > -48]

condition_3 = (cohort_df.OUT_DIS_DIFF < 0)
cohort_df.loc[condition_3, 'OUTTIME']  = cohort_df.loc[condition_3, 'DISCHTIME']

In [34]:
cohort_df.drop(columns=['OUT_DIS_DIFF', 'ADM_IN_DIFF'], inplace=True)

In [35]:
cohort_df = cohort_df[(cohort_df.ICUSTAY_ID.notnull()) & (cohort_df.INTIME >= cohort_df.ADMITTIME) & 
                      (cohort_df.OUTTIME <= cohort_df.DISCHTIME)]

In [36]:
cohort_df['ICU_EXPIRE_FLAG'] = 0
cohort_df.loc[((cohort_df['DEATHTIME'] > cohort_df['INTIME']) & (cohort_df['DEATHTIME'] < cohort_df['OUTTIME'] 
                                                              + pd.Timedelta(hours=6))), 'ICU_EXPIRE_FLAG'] = 1

In [37]:
cohort_df['ICU_LOS_H'] = cohort_df['OUTTIME'] - cohort_df['INTIME']
cohort_df['ICU_LOS_H'] = pd.to_timedelta(cohort_df['ICU_LOS_H'])

cohort_df['ICU_LOS_H'] = (cohort_df.ICU_LOS_H / pd.Timedelta(hours = 1)).round(1)
cohort_df.rename(columns={"LOS": "ICU_LOS_D"}, inplace=True)

In [38]:
cohort_df['HOSP_LOS'] = cohort_df['DISCHTIME'] - cohort_df['ADMITTIME']
cohort_df['HOSP_LOS'] = pd.to_timedelta(cohort_df['HOSP_LOS'])

cohort_df['HOSP_LOS_D'] =  (cohort_df.HOSP_LOS / pd.Timedelta(days = 1)).round(1)
cohort_df['HOSP_LOS_H'] =  (cohort_df.HOSP_LOS / pd.Timedelta(hours= 1)).round(1)

cohort_df.drop(columns=['HOSP_LOS'], inplace=True)

In [39]:
cohort_df = cohort_df[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'GENDER', 'AGE', 'ETHNICITY', 'INTIME', 'OUTTIME',
                       'ADMITTIME', 'DISCHTIME', 'ICU_LOS_H', 'ICU_LOS_D', 'HOSP_LOS_H', 'HOSP_LOS_D',
                       'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'EXPIRE_FLAG', 'DEATH_TIME_DISCH']]

In [40]:
cohort_df.head(3)

In [41]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df.ICUSTAY_ID.nunique())
print(cohort_df[cohort_df.ICU_EXPIRE_FLAG == 1].ICUSTAY_ID.nunique())
print(cohort_df[cohort_df.HOSPITAL_EXPIRE_FLAG == 1].ICUSTAY_ID.nunique())
print(cohort_df[cohort_df.EXPIRE_FLAG == 1].ICUSTAY_ID.nunique())

15100
18070
19351
1514
2169
6218


### Final cohort

In [42]:
cohort_icu_df = cohort_df[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'INTIME', 'OUTTIME']]

In [43]:
cohort_icu_df.head(3)

### Prescription

In [44]:
prescrip = []

for chunk in pd.read_csv(mimiciii + "PRESCRIPTIONS.csv", chunksize=10000):
    chunk = chunk[chunk.HADM_ID.isin(cohort_hadm_id_icu)]
    chunk.drop(columns=['ROW_ID', 'DRUG_NAME_POE', 'DRUG_NAME_GENERIC',
                        'FORMULARY_DRUG_CD', 'NDC', 'PROD_STRENGTH', 'FORM_VAL_DISP', 'FORM_UNIT_DISP'], 
               inplace=True)
    if chunk.shape[0] > 0:
        prescrip.append(chunk)
        
prescriptions = pd.concat(prescrip)

In [45]:
antibiotic = prescriptions.copy()
antibiotic = antibiotic[antibiotic.DRUG_TYPE != 'BASE']
antibiotic['DRUG'] = antibiotic['DRUG'].apply(str.lower)
antibiotic = antibiotic[~antibiotic.ROUTE.isin(['OU', 'OS', 'OD', 'AU', 'AS', 'AD', 'TP'])]
antibiotic = antibiotic[antibiotic.ROUTE.notnull()]
antibiotic['ROUTE'] = antibiotic['ROUTE'].apply(str.lower)
antibiotic = antibiotic[~antibiotic["ROUTE"].str.contains('eye|ear')]
antibiotic = antibiotic[~antibiotic["DRUG"].str.contains('cream|desensitization|ophth oint|gel')]

In [46]:
substring_antibiotics = ['adoxa', 'ala-tet', 'alodox', 'amikacin', 'amikin', 'amoxicill', 'amphotericin', 
                         'anidulafungin', 'ancef', 'clavulanate', 'ampicillin', 'augmentin', 'avelox', 'avidoxy', 
                         'azactam', 'azithromycin', 'aztreonam', 'axetil', 'bactocill', 'bactrim', 'bactroban', 
                         'bethkis', 'biaxin', 'bicillin l-a', 'cayston', 'cefazolin', 'cedax', 'cefoxitin', 
                         'ceftazidime', 'cefaclor', 'cefadroxil', 'cefdinir', 'cefditoren', 'cefepime', 'cefotan',
                         'cefotetan', 'cefotaxime', 'ceftaroline', 'cefpodoxime', 'cefpirome', 'cefprozil', 
                         'ceftibuten', 'ceftin', 'ceftriaxone', 'cefuroxime', 'cephalexin', 'cephalothin', 
                         'cephapririn', 'chloramphenicol', 'cipro', 'ciprofloxacin', 'claforan', 'clarithromycin',
                         'cleocin', 'clindamycin', 'cubicin', 'dicloxacillin', 'dirithromycin', 'doryx', 'doxycy',
                         'duricef', 'dynacin', 'ery-tab', 'eryped', 'eryc', 'erythrocin', 'erythromycin', 
                         'factive', 'flagyl', 'fortaz', 'furadantin', 'garamycin', 'gentamicin', 'kanamycin', 
                         'keflex', 'kefzol', 'ketek', 'levaquin', 'levofloxacin', 'lincocin', 'linezolid', 
                         'macrobid', 'macrodantin', 'maxipime', 'mefoxin', 'metronidazole', 'meropenem', 
                         'methicillin', 'minocin', 'minocycline', 'monodox', 'monurol', 'morgidox', 'moxatag', 
                         'moxifloxacin', 'mupirocin', 'myrac', 'nafcillin', 'neomycin', 'nicazel doxy 30',
                         'nitrofurantoin', 'norfloxacin', 'noroxin', 'ocudox', 'ofloxacin', 'omnicef', 'oracea', 
                         'oraxyl', 'oxacillin', 'pc pen vk', 'pce dispertab', 'panixine', 'pediazole', 'penicillin',
                         'periostat', 'pfizerpen', 'piperacillin', 'tazobactam', 'primsol', 'proquin', 'raniclor',
                         'rifadin', 'rifampin', 'rocephin', 'smz-tmp', 'septra', 'septra ds', 'septra', 'solodyn',
                         'spectracef', 'streptomycin', 'sulfadiazine', 'sulfamethoxazole', 'trimethoprim', 
                         'sulfatrim', 'sulfisoxazole', 'suprax', 'synercid', 'tazicef', 'tetracycline', 'timentin',
                         'tobramycin', 'trimethoprim', 'unasyn', 'vancocin', 'vancomycin', 'vantin', 'vibativ', 
                         'vibra-tabs', 'vibramycin', 'zinacef', 'zithromax', 'zosyn', 'zyvox']

In [47]:
gsn_antibiotics = [ '002542','002543','007371','008873','008877','008879','008880','008935',
                    '008941','008942','008943','008944','008983','008984','008990','008991',
                    '008992','008995','008996','008998','009043','009046','009065','009066',
                    '009136','009137','009162','009164','009165','009171','009182','009189',
                    '009213','009214','009218','009219','009221','009226','009227','009235',
                    '009242','009263','009273','009284','009298','009299','009310','009322',
                    '009323','009326','009327','009339','009346','009351','009354','009362',
                    '009394','009395','009396','009509','009510','009511','009544','009585',
                    '009591','009592','009630','013023','013645','013723','013724','013725',
                    '014182','014500','015979','016368','016373','016408','016931','016932',
                    '016949','018636','018637','018766','019283','021187','021205','021735',
                    '021871','023372','023989','024095','024194','024668','025080','026721',
                    '027252','027465','027470','029325','029927','029928','037042','039551',
                    '039806','040819','041798','043350','043879','044143','045131','045132',
                    '046771','047797','048077','048262','048266','048292','049835','050442',
                    '050443','051932','052050','060365','066295','067471']

In [48]:
antibiotic['VALUE'] = 0

for string in substring_antibiotics:
    antibiotic.loc[antibiotic['DRUG'].str.contains(string),  'VALUE'] = 1
    
antibiotic.loc[antibiotic['GSN'].isin(gsn_antibiotics),  'VALUE'] = 1

antibiotic.loc[antibiotic.VALUE == 1,  'DRUG'] = 'Antibiotic_PRC'
antibiotic = antibiotic[antibiotic.VALUE == 1]

In [49]:
antibiotic.drop(columns=['GSN']   , inplace=True)
prescriptions.drop(columns=['GSN'], inplace=True)

In [50]:
antibiotic['STARTDATE'] = pd.to_datetime(antibiotic['STARTDATE'])
antibiotic['ENDDATE']   = pd.to_datetime(antibiotic['ENDDATE'])

In [51]:
antibiotic = pd.merge(antibiotic, cohort_icu_df, on=['SUBJECT_ID', 'HADM_ID'], how='left')
antibiotic = antibiotic[(antibiotic['STARTDATE'] >= antibiotic['INTIME']) & (antibiotic['STARTDATE'] <= antibiotic['OUTTIME'])]
antibiotic.loc[(antibiotic['ENDDATE'] > antibiotic['OUTTIME']), 'ENDDATE'] =  antibiotic['OUTTIME']
antibiotic = antibiotic[antibiotic.ICUSTAY_ID_x == antibiotic.ICUSTAY_ID_y]

In [52]:
Fentanyl = ['Fentanyl Citrate', 'Fentanyl Patch', 'NEO*IV*Fentanyl', 'Fentanyl PCA', 'Fentanyl', 'FENTANYL',
            'fentaNYL', 'fentaNYL citrate (PF)', 'Fentanyl ']

Propofol = ['Propofol', 'Propofol (Generic)', 'PROPOFOL', 'Propofol (Diprivan)', 'PROPOFOL (*GENERIC*)',
            'Propofol Diprivan', 'PROPOF', 'propo']

Norepinephrine = ['Norepinephrine', 'NORepinephrine', 'Norepinephrine Bitartrate']

Insulin = ['Insulin', 'Insulin Human Regular', 'Humulin-R Insulin', 'Insulin Regular Human (U-500)',
           'Insulin Pump', 'Insulin Pump (Self Administering Medication)', 'NEO*IV*Insulin (Dilute)',
           'Insulin Human NPH', 'Insulin Glargine', 'Insulin Detemir', 'Humalog Insulin', 'Insulin Aspart *NF*',
           'insulin', 'Regular Insulin', 'Insulin Human 70/30', 'insulin detemir', 'Insulin NPH Human Recomb', 
           'Insulin Regular Human', 'Insulin Reg Human (NovoLIN R)', 'Insulin Human Regular (Dilute)',
           'Humulin-R Insulin (U-500)', 'Insulin ', 'Insulin Lispro 75/25', 'Insulin - Sliding Scale',
           'Insulin Humalog 75/25 Pen', 'Insulin Levemir', 'INSULIN ULTRALENTE', 'HumaLOG Insulin',
           'Insulin Regular Pork (Iletin II)', 'Ultralente Humulin Insulin', 'Insulin Lispro (Human)', 'insuli',
           'regular insulin', 'insulin dum', 'insul', 'insulin ', 'Insulin Aspart', 'Insulin novolog',
           'Insulin Aspart *NF* (for Insulin PUMP)', 'HUMALOG INSULIN', 'Insulin ASPART', 
           'NPH insulin human recomb', 'insulin pump', 'Insulin Lispro Desensitization Protocol']

Midazolam = ['Midazolam', 'Midazolam HCl', 'NEO*PO*Midazolam', 'MIDAZOLAM']

Heparin = ['Heparin', 'Heparin Sodium', 'Heparin Flush (10 units/ml)', 'Heparin Flush CVL  (100 units/ml)',
           'Heparin Flush PICC (100 units/ml)', 'Heparin (Preservative Free)', 'Heparin Dwell (1000 Units/mL)',
           'Heparin Flush (100 units/ml)', 'Heparin Flush (1000 units/mL)', 'Heparin Sodium (Preservative Free)',
           'Heparin Flush (5000 Units/mL)', 'Heparin Flush', 'Heparin Flush Port (10units/ml)',
           'Heparin (CRRT Machine Priming)', 'Heparin Flush CRRT (5000 Units/mL)', 
           'Heparin Flush Midline (100 units/ml)', 'Heparin CRRT', 'Heparin (IABP)', 'Heparin (Hemodialysis)',
           'Heparin Flush Port (10 units/mL)', 'Heparin Flush Hickman (100 units/ml)', 'Heparin ',
           'Heparin Lock Flush', 'Heparin Flush CVL  (100 units/ml) ', 'Heparin (Preserv. Free)', 'Heparin Flush ',
           'heparin', 'HEPARIN', 'Hepar', 'Hepari', 'Heparin Flush (10 Units/mL)', 'HEParin (Porcine) in NS (PF)',
           'hepar', 'Hepa', 'Heparin (Porcine)', 'Heparin Flush (100 units/mL)', 'HEPARIN  FLUSH', 'Heparin Flus',
           'Heparin flush', 'Heparin Flush (1000 units/ml)', 'Heparin Flush (10 units/mL)', 'Heparin Flush Port ',
           'Heparin Flu', 'Heparin lock']

Dexmedetomidine = ['Dexmedetomidine', 'Dexmedetomidine HCl', 'Dexmedetomidine Hcl']

Amiodarone = ['Amiodarone', 'Amiodarone HCl', 'AMIODARONE', 'Amiod', 'Amiodarone Oral Suspension']

Vasopressin = ['Vasopressin']

Phenylephrine = ['Phenylephrine', 'Phenylephrine HCl', 'PHENYLEPHrine', 'Cyclopentolate-Phenylephrine',
                 'NEO*NASAL*Phenylephrine 0.125%', 'Phenylephrine 2.5 % Ophth Soln', 
                 'Phenylephrine  0.5% Nasal Spray', 'PHENYLEPHRINE HCL', 'Phenylephrine 2.5 %',
                 'Phenylephrine HCl 1%', 'phenylephrine HCl', 'Phenylephrine 2.5% opth', 
                 'Phenyleprhine Ophth Soln 10%', 'Phenylephrine 1% Nasal Spray', 
                 'Phenylephrine  0.05% Nasal Spray']

Epinephrine = ['Epinephrine', 'Epinephrine 1:1000', 'EPINEPHrine', 'Lidocaine 1%/Epinephrine 1:100000',
               'Epinephrine Inhalation', 'Epinephrine HCl', 'Lidocaine 0.5%/Epinephrine',
               'Lidocaine 2%/Epinephrine', 'Lidocaine 2%/Epinephrine P.F.', 'Epinephrine ', 
               'Bupivacaine 0.25%-Epinephrine', 'Bupivacaine 0.50%-Epinephrine', 'Epinephrine-Sodium Chloride',
               'EPINEPHRINE', 'Epinephrin', 'Epinephrine Kit', 'epinephrine', 'Lidocaine 1%/Epinephrine',
               'Epinephri', 'Epineph', 'Lidocaine 1.5%/Epinephrine P.F.', 'racemic epinephrine',
               'EPINEPHrine Auto Injector', 'Lidocaine 1%/Epinephrine 1:100,000', 'Epinephrine Auto Injector',
               'Epinephrine Base', 'Epinephrine Topical Soln']

Dopamine = ['DopAmine', 'DOPamine', 'Dopamine HCl', 'Dopamine Hcl', 'Dopamine']

Nicardipine = ['NiCARdipine IV', 'Nicardipine HCl IV', 'NiCARdipine', '*NF* Nicardipine HCl IV', 'Nicardipine HCl',
               'Nicardipine']

Milrinone = ['Milrinone', 'Milrinone Lactate']

Pantoprazole = ['Pantoprazole', 'Pantoprazole Sodium', 'Pantoprazole (Self Med)', 'panto']

Diltiazem = ['Diltiazem', 'Diltiazem Extended-Release', 'diltia', '*NF* Diltiazem SR', 'diltiaz', 'Diltia',
             'diltiazem', 'Diltiazem ', 'Diltiaz', 'Dilti']

Dobutamine = ['DOBUTamine', 'Dobutamine', 'Dobutamine HCl', 'Dobutamine Hcl']

Nitroglycerin = ['Nitroglycerin', 'Nitroglycerin SL', 'Nitroglycerin Ointment  2%', 'Nitroglycerin Oint. 2%',
                 'Nitroglycerin Patch', 'Nitroglycerin SR', 'Nitroglycerin ', 'NITROGLYCERIN', 
                 '*NF* Nitroglycerin Ointment']

Warfarin    = ['Warfarin', 'Warf', 'Warfar', '*NF* Warfarin', 'warfarin', 'warf', 
               'warfarin (Coumadin) Brand Name', 'Coumadin']

Apixaban    = ['Apixaban', 'INV-Apixaban', 'apixaban', 'INV apixaban']

Dabigatran  = ['Dabigatran Etexilate', 'dabigatran etexilate', 'Dabigatran', 'dabigatran etexilate (Pradaxa)', 
               'Pradaxa']

Rivaroxaban = ['Rivaroxaban', 'rivaroxaban', 'INV-Rivaroxaban', 'Rivaro', 'Xarelto']

Edoxaban    = ['edoxaban', 'INV-Edoxaban', 'Edoxaban ', 'Edoxaban (Savaysa) 60mg tab', 'Edoxaban', 
               'INV-edoxaban', 'Edoxaban 60mg', 'Edoxiban']

In [53]:
prescriptions = prescriptions[prescriptions.DRUG.isin(Fentanyl)        | prescriptions.DRUG.isin(Propofol)     | 
                              prescriptions.DRUG.isin(Norepinephrine)  | prescriptions.DRUG.isin(Insulin)      | 
                              prescriptions.DRUG.isin(Midazolam)       | prescriptions.DRUG.isin(Heparin)      | 
                              prescriptions.DRUG.isin(Dexmedetomidine) | prescriptions.DRUG.isin(Amiodarone)   | 
                              prescriptions.DRUG.isin(Vasopressin)     | prescriptions.DRUG.isin(Phenylephrine)|
                              prescriptions.DRUG.isin(Dopamine)        | prescriptions.DRUG.isin(Nicardipine)  |
                              prescriptions.DRUG.isin(Milrinone)       | prescriptions.DRUG.isin(Pantoprazole) |
                              prescriptions.DRUG.isin(Diltiazem)       | prescriptions.DRUG.isin(Dobutamine)   |
                              prescriptions.DRUG.isin(Nitroglycerin)   | prescriptions.DRUG.isin(Epinephrine)  |
                              prescriptions.DRUG.isin(Warfarin)        | prescriptions.DRUG.isin(Apixaban)     | 
                              prescriptions.DRUG.isin(Dabigatran)      | prescriptions.DRUG.isin(Rivaroxaban)  |
                              prescriptions.DRUG.isin(Edoxaban)]

In [54]:
prescriptions.loc[prescriptions.DRUG.isin(Fentanyl)  ,       'DRUG'] = 'Fentanyl_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Propofol)  ,       'DRUG'] = 'Propofol_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Norepinephrine)  , 'DRUG'] = 'Norepinephrine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Insulin)  ,        'DRUG'] = 'Insulin_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Midazolam)  ,      'DRUG'] = 'Midazolam_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Heparin)  ,        'DRUG'] = 'Heparin_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Dexmedetomidine),  'DRUG'] = 'Dexmedetomidine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Amiodarone)  ,     'DRUG'] = 'Amiodarone_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Vasopressin)  ,    'DRUG'] = 'Vasopressin_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Phenylephrine)  ,  'DRUG'] = 'Phenylephrine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Dopamine)  ,       'DRUG'] = 'Dopamine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Nicardipine)  ,    'DRUG'] = 'Nicardipine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Milrinone)  ,      'DRUG'] = 'Milrinone_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Pantoprazole)  ,   'DRUG'] = 'Pantoprazole_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Diltiazem)  ,      'DRUG'] = 'Diltiazem_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Dobutamine)  ,     'DRUG'] = 'Dobutamine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Nitroglycerin)  ,  'DRUG'] = 'Nitroglycerin_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Epinephrine)  ,    'DRUG'] = 'Epinephrine_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Warfarin),         'DRUG'] = 'Warfarin_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Apixaban),         'DRUG'] = 'Apixaban_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Dabigatran),       'DRUG'] = 'Dabigatran_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Rivaroxaban),      'DRUG'] = 'Rivaroxaban_PRC'
prescriptions.loc[prescriptions.DRUG.isin(Edoxaban),         'DRUG'] = 'Edoxaban_PRC'

In [55]:
prescriptions['STARTDATE'] = pd.to_datetime(prescriptions['STARTDATE'])
prescriptions['ENDDATE']   = pd.to_datetime(prescriptions['ENDDATE'])

In [56]:
prescriptions = pd.merge(prescriptions, cohort_icu_df, on=['SUBJECT_ID', 'HADM_ID'], how='left')
prescriptions = prescriptions[(prescriptions['STARTDATE'] >= prescriptions['INTIME']) & (prescriptions['STARTDATE'] <= prescriptions['OUTTIME'])]
prescriptions.loc[(prescriptions['ENDDATE'] > prescriptions['OUTTIME']), 'ENDDATE'] =  prescriptions['OUTTIME']
prescriptions = prescriptions[prescriptions.ICUSTAY_ID_x == prescriptions.ICUSTAY_ID_y]

In [57]:
antibiotic = antibiotic[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID_x', 'STARTDATE', 'ENDDATE', 'DRUG']]
antibiotic.rename(columns={"DRUG": "LABEL", "ICUSTAY_ID_x": "ICUSTAY_ID"}, inplace=True)

prescriptions = prescriptions[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID_x', 'STARTDATE', 'ENDDATE', 'DRUG']]
prescriptions.rename(columns={"DRUG": "LABEL", "ICUSTAY_ID_x": "ICUSTAY_ID"}, inplace=True)

In [58]:
antibiotic['Hour_Diff'] = (antibiotic.ENDDATE - antibiotic.STARTDATE).apply(lambda s: s / np.timedelta64(1, 's')) / 60./60
prescriptions['Hour_Diff'] = (prescriptions.ENDDATE - prescriptions.STARTDATE).apply(lambda s: s / np.timedelta64(1, 's')) / 60./60

antibiotic.loc[antibiotic.Hour_Diff < 0,    'ENDDATE']  = antibiotic['STARTDATE']
antibiotic.loc[antibiotic.Hour_Diff < 0,    'Hour_Diff'] = 0

prescriptions.loc[prescriptions.Hour_Diff < 0,    'ENDDATE']  = antibiotic['STARTDATE']
prescriptions.loc[prescriptions.Hour_Diff < 0,    'Hour_Diff'] = 0

antibiotic.drop(columns=['Hour_Diff', 'ENDDATE']   , inplace=True)
prescriptions.drop(columns=['Hour_Diff', 'ENDDATE'], inplace=True)

antibiotic['VALUE'] = 1
prescriptions['VALUE'] = 1

In [59]:
prescriptions_list = [antibiotic, prescriptions]
prescriptions = pd.concat(prescriptions_list)
prescriptions.columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE']

In [60]:
prescriptions.head(2)

In [61]:
print(prescriptions.SUBJECT_ID.nunique())
print(prescriptions.HADM_ID.nunique())
print(prescriptions.ICUSTAY_ID.nunique())

12690
14969
15880


### microbiology events

In [62]:
microbioevents = []

for chunk in pd.read_csv(mimiciii + "MICROBIOLOGYEVENTS.csv", chunksize=10000):
    chunk = chunk[chunk.SUBJECT_ID.isin(cohort_subject_id_icu)]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CHARTTIME', 'AB_NAME','INTERPRETATION']]
    if chunk.shape[0] > 0:
        microbioevents.append(chunk)
        
microbio_df = pd.concat(microbioevents)

In [63]:
microbio_df['CHARTDATE'] = pd.to_datetime(microbio_df['CHARTDATE'])
microbio_df['CHARTTIME'] = pd.to_datetime(microbio_df['CHARTTIME'])

In [64]:
microbio_df.loc[microbio_df['CHARTTIME'].isnull(),  'CHARTTIME'] = microbio_df['CHARTDATE']
microbio_df.loc[microbio_df['AB_NAME'].isnull(),  'AB_NAME'] = 'NO GROWTH'
microbio_df.loc[microbio_df['INTERPRETATION'].isnull(),  'INTERPRETATION'] = 'N'
microbio_df.drop(columns=['CHARTDATE'], inplace=True)

In [65]:
microbio_df = pd.merge(microbio_df, cohort_icu_df, on=['SUBJECT_ID'], how='left')

microbio_df = microbio_df[(microbio_df['CHARTTIME'] >= microbio_df['INTIME']) & 
                          (microbio_df['CHARTTIME'] <= microbio_df['OUTTIME'])]

In [66]:
microbio_df = microbio_df[['SUBJECT_ID', 'HADM_ID_y', 'ICUSTAY_ID', 'CHARTTIME', 'AB_NAME','INTERPRETATION']]
microbio_df.rename(columns={"HADM_ID_y":"HADM_ID", "AB_NAME": "LABEL", "INTERPRETATION": "VALUE"}, inplace=True)

In [67]:
microbio_df['LABEL'] = 'Microbio Test'
microbio_df.loc[microbio_df['VALUE'] == 'N'  , 'VALUE'] = -1
microbio_df.loc[microbio_df['VALUE'] == 'S'  , 'VALUE'] = 4
microbio_df.loc[microbio_df['VALUE'] == 'R'  , 'VALUE'] = 3
microbio_df.loc[microbio_df['VALUE'] == 'I'  , 'VALUE'] = 2
microbio_df.loc[microbio_df['VALUE'] == 'P'  , 'VALUE'] = 1

In [68]:
microbio_df.head(3)

In [69]:
print(microbio_df.SUBJECT_ID.nunique())
print(microbio_df.HADM_ID.nunique())
print(microbio_df.ICUSTAY_ID.nunique())

13837
16279
17240


### lab item ids

In [70]:
d_labitems = pd.read_csv(mimiciii + "D_LABITEMS.csv")
d_labitems.drop(columns=['ROW_ID', 'FLUID', 'CATEGORY', 'LOINC_CODE'], inplace=True)

### lab event

In [71]:
def check(x):
    try:
        x = float(str(x).strip())
    except ValueError:
        try:
            x = float(re.findall(r'\d+', x)[0])
        except IndexError:
            x = np.nan
    return x

def check_itemvalue(df):
    df['VALUE'] = df['VALUE'].apply(lambda x: check(x))
    return df

In [72]:
labevents = []

for chunk in pd.read_csv(mimiciii + "LABEVENTS.csv", chunksize=10000):
    chunk = chunk[chunk.HADM_ID.isin(cohort_hadm_id_icu)]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM', 'VALUEUOM']]
    chunk = chunk[chunk.ITEMID.notnull()]
    chunk = chunk[(chunk.VALUE.notnull()) | (chunk.VALUENUM.notnull())]
    if chunk.shape[0] > 0:
        labevents.append(chunk)
        
lab_df = pd.concat(labevents)

In [73]:
lab_df = pd.merge(lab_df, d_labitems, on=['ITEMID'], how='left')
lab_df = pd.merge(lab_df, cohort_icu_df, on=['SUBJECT_ID', 'HADM_ID'], how='left')

In [74]:
lab_df['CHARTTIME'] = pd.to_datetime(lab_df['CHARTTIME'])
lab_df = lab_df[(lab_df['CHARTTIME'] >= lab_df['INTIME']) & (lab_df['CHARTTIME'] <= lab_df['OUTTIME'])]

In [75]:
lab_df = lab_df[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE', 'VALUENUM', 'VALUEUOM']]

In [76]:
lab_df['string'] = lab_df['VALUE'].str.extract('([A-Za-z]+)')
lab_df.loc[(lab_df.string.notnull()) & (lab_df.VALUENUM.notnull()), 'VALUE'] = lab_df['VALUENUM']
lab_df['string'] = lab_df['VALUE'].str.extract('([A-Za-z]+)')

In [77]:
lab_df = check_itemvalue(lab_df)

In [78]:
lab_df.loc[(lab_df.VALUE != lab_df.VALUENUM) & (lab_df.VALUE.isnull()) & (lab_df.VALUENUM.notnull()), 'VALUE'] = lab_df['VALUENUM']
lab_df.loc[((lab_df.string.notnull()) & (lab_df.VALUE.isnull())), 'VALUE'] = lab_df['string']
lab_df.loc[(lab_df['LABEL'] == 'Absolute Lymphocyte Count') & (lab_df['VALUENUM'] == '#/uL'), 'VALUE'] =  np.nan

In [79]:
Intubated_0 = ['NOT']
Intubated_1 = ['INTUBATED']

lab_df.loc[(lab_df['LABEL'] == 'Intubated') & (lab_df['VALUE'] != 'NOT') & (lab_df['VALUE'] != 'INTUBATED'),  'VALUE'] = np.nan
lab_df.loc[(lab_df['LABEL'] == 'Intubated') & (lab_df['VALUE'].isin(Intubated_0)),  'VALUE'] = 0
lab_df.loc[(lab_df['LABEL'] == 'Intubated') & (lab_df['VALUE'].isin(Intubated_1)),  'VALUE'] = 1

In [80]:
lab_df = lab_df[lab_df.LABEL.notnull()]
lab_df = lab_df[lab_df.VALUE.notnull()]
lab_df.drop(columns=['VALUENUM', 'string', 'VALUEUOM'], inplace=True)

In [81]:
lab_df.head(3)

In [82]:
print(lab_df.SUBJECT_ID.nunique())
print(lab_df.HADM_ID.nunique())
print(lab_df.ICUSTAY_ID.nunique())

14827
17711
18921


### D-items

In [83]:
d_icuitems = pd.read_csv(mimiciii + "D_ITEMS.csv")
d_icuitems.drop(columns=['ROW_ID', 'ABBREVIATION', 'DBSOURCE', 'LINKSTO', 'CATEGORY', 'UNITNAME',
                         'PARAM_TYPE', 'CONCEPTID'], inplace=True)

### chartevent

In [84]:
chartevent = []

for chunk in pd.read_csv(mimiciii + "CHARTEVENTS.csv", chunksize=10000):
    chunk = chunk[chunk.ICUSTAY_ID.isin(cohort_stay_id_icu)]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM', 'VALUEUOM']]
    chunk = chunk[chunk.ITEMID.notnull()]
    chunk = chunk[(chunk.VALUE.notnull()) | (chunk.VALUENUM.notnull())]
    if chunk.shape[0] > 0:
        chartevent.append(chunk)
        
chart_df = pd.concat(chartevent)

In [85]:
chart_df = pd.merge(chart_df, d_icuitems, on=['ITEMID'], how='left')
chart_df['CHARTTIME'] = pd.to_datetime(chart_df['CHARTTIME'])
chart_df = chart_df[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE', 'VALUENUM', 'VALUEUOM']]

In [86]:
chart_df['string'] = chart_df['VALUE'].str.extract('([A-Za-z]+)')
chart_df.loc[(chart_df.string.notnull()) & (chart_df.VALUENUM.notnull()), 'VALUE'] = chart_df['VALUENUM']
chart_df['string'] = chart_df['VALUE'].str.extract('([A-Za-z]+)')
chart_df.loc[(chart_df.string.notnull()), 'string'] = chart_df['VALUE']

In [87]:
chart_df = check_itemvalue(chart_df)
chart_df.loc[(chart_df.VALUE != chart_df.VALUENUM) & (chart_df.VALUE.isnull()) & (chart_df.VALUENUM.notnull()), 'VALUE'] = chart_df['VALUENUM']
chart_df.loc[((chart_df.string.notnull()) & (chart_df.VALUE.isnull())), 'VALUE'] = chart_df['string']

In [88]:
MentalStatus_1 = [15.0]

chart_df.loc[(chart_df['LABEL'] == 'Mental status') & (chart_df['VALUE'] != 0.0) & (chart_df['VALUE'] != 15.0),  'VALUE'] = np.nan
chart_df.loc[(chart_df['LABEL'] == 'Mental status') & (chart_df['VALUE'].isin(MentalStatus_1)),    'VALUE'] = 1.0

In [89]:
CAMICU_Disorganized_thinking_0 = ['No']
CAMICU_Disorganized_thinking_1 = ['Yes']

chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU Disorganized thinking') & (chart_df['VALUE'].isin(CAMICU_Disorganized_thinking_0)),    'VALUE'] = 0
chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU Disorganized thinking') & (chart_df['VALUE'].isin(CAMICU_Disorganized_thinking_1)),    'VALUE'] = 1

In [90]:
CAMICU_RASS_LOC_0 = ['No']
CAMICU_RASS_LOC_1 = ['Yes']

chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU RASS LOC') & (chart_df['VALUE'] != 'No') & (chart_df['VALUE'] != 'Yes'),  'VALUE'] = np.nan
chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU RASS LOC') & (chart_df['VALUE'].isin(CAMICU_RASS_LOC_0)),    'VALUE'] = 0
chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU RASS LOC') & (chart_df['VALUE'].isin(CAMICU_RASS_LOC_1)),    'VALUE'] = 1

In [91]:
CAMICU_MS_change_0 = ['No (Stop - Not delirious)']
CAMICU_MS_change_1 = ['Yes (Continue)']
CAMICU_MS_change_nan = ['Unable to Assess (Stop)']

chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU MS change') & (chart_df['VALUE'] != 'No (Stop - Not delirious)') & 
             (chart_df['VALUE'] != 'Yes (Continue)'),  'VALUE'] = np.nan
chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU MS change') & (chart_df['VALUE'].isin(CAMICU_MS_change_0)),   'VALUE'] = 0
chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU MS change') & (chart_df['VALUE'].isin(CAMICU_MS_change_1)),   'VALUE'] = 1
chart_df.loc[(chart_df['LABEL'] == 'CAM-ICU MS change') & (chart_df['VALUE'].isin(CAMICU_MS_change_nan)), 'VALUE'] = np.nan

In [92]:
SkinTemperature_1 = ['Cold']
SkinTemperature_2 = ['Cool']
SkinTemperature_3 = ['Warm']
SkinTemperature_4 = ['Hot']
SkinTemperature_nan = ['Other/Remarks']

chart_df.loc[(chart_df['LABEL'] == 'Skin Temperature') & (chart_df['VALUE'] != 'Cold') & 
             (chart_df['VALUE'] != 'Cool') & (chart_df['VALUE'] != 'Warm') & (chart_df['VALUE'] != 'Hot'),  'VALUE'] = np.nan

chart_df.loc[(chart_df['LABEL'] == 'Skin Temperature') & (chart_df['VALUE'].isin(SkinTemperature_1)), 'VALUE'] = 1
chart_df.loc[(chart_df['LABEL'] == 'Skin Temperature') & (chart_df['VALUE'].isin(SkinTemperature_2)), 'VALUE'] = 2
chart_df.loc[(chart_df['LABEL'] == 'Skin Temperature') & (chart_df['VALUE'].isin(SkinTemperature_3)), 'VALUE'] = 3
chart_df.loc[(chart_df['LABEL'] == 'Skin Temperature') & (chart_df['VALUE'].isin(SkinTemperature_4)), 'VALUE'] = 4

chart_df.loc[(chart_df['LABEL'] == 'Skin [Temperature]') & (chart_df['VALUE'] != 'Cold') & 
             (chart_df['VALUE'] != 'Cool') & (chart_df['VALUE'] != 'Warm') & (chart_df['VALUE'] != 'Hot'),  'VALUE'] = np.nan

chart_df.loc[(chart_df['LABEL'] == 'Skin [Temperature]') & (chart_df['VALUE'].isin(SkinTemperature_1)),   'VALUE'] = 1
chart_df.loc[(chart_df['LABEL'] == 'Skin [Temperature]') & (chart_df['VALUE'].isin(SkinTemperature_2)),   'VALUE'] = 2
chart_df.loc[(chart_df['LABEL'] == 'Skin [Temperature]') & (chart_df['VALUE'].isin(SkinTemperature_3)),   'VALUE'] = 3
chart_df.loc[(chart_df['LABEL'] == 'Skin [Temperature]') & (chart_df['VALUE'].isin(SkinTemperature_4)),   'VALUE'] = 4
chart_df.loc[(chart_df['LABEL'] == 'Skin [Temperature]') & (chart_df['VALUE'].isin(SkinTemperature_nan)), 'VALUE'] = np.nan

In [93]:
PainPresent_0 = ['No']
PainPresent_1 = ['Yes']
PainPresent_nan = ['Other/Remarks']

chart_df.loc[(chart_df['LABEL'] == 'Pain Present') & (chart_df['VALUE'] != 'No') & (chart_df['VALUE'] != 'Yes'),  'VALUE'] = np.nan
chart_df.loc[(chart_df['LABEL'] == 'Pain Present') & (chart_df['VALUE'].isin(PainPresent_0)),  'VALUE'] = 0
chart_df.loc[(chart_df['LABEL'] == 'Pain Present') & (chart_df['VALUE'].isin(PainPresent_1)),  'VALUE'] = 1
chart_df.loc[(chart_df['LABEL'] == 'Pain Present') & (chart_df['VALUE'].isin(PainPresent_nan)),'VALUE'] = np.nan

In [94]:
RiskFalls_0 = ['No']
RiskFalls_1 = ['Yes']
RiskFalls_nan = ['Other/Remarks']

chart_df.loc[(chart_df['LABEL'] == 'Risk for Falls') & (chart_df['VALUE'] != 'No') & (chart_df['VALUE'] != 'Yes'),  'VALUE'] = np.nan
chart_df.loc[(chart_df['LABEL'] == 'Risk for Falls') & (chart_df['VALUE'].isin(RiskFalls_0)),  'VALUE'] = 0
chart_df.loc[(chart_df['LABEL'] == 'Risk for Falls') & (chart_df['VALUE'].isin(RiskFalls_1)),  'VALUE'] = 1
chart_df.loc[(chart_df['LABEL'] == 'Risk for Falls') & (chart_df['VALUE'].isin(RiskFalls_nan)),'VALUE'] = np.nan

In [95]:
DeliriumAssess_0 = ['Negative']
DeliriumAssess_1 = ['Positive']
DeliriumAssess_nan = ['UTA']

chart_df.loc[(chart_df['LABEL'] == 'Delirium assessment') & (chart_df['VALUE'] != 'Negative') & (chart_df['VALUE'] != 'Positive'),  'VALUE'] = np.nan
chart_df.loc[(chart_df['LABEL'] == 'Delirium assessment') & (chart_df['VALUE'].isin(DeliriumAssess_0)),  'VALUE'] = 0
chart_df.loc[(chart_df['LABEL'] == 'Delirium assessment') & (chart_df['VALUE'].isin(DeliriumAssess_1)),  'VALUE'] = 1
chart_df.loc[(chart_df['LABEL'] == 'Delirium assessment') & (chart_df['VALUE'].isin(DeliriumAssess_nan)),'VALUE'] = np.nan

In [96]:
chart_df = chart_df[chart_df.LABEL.notnull()]
chart_df = chart_df[chart_df.VALUE.notnull()]
chart_df.drop(columns=['VALUENUM', 'string', 'VALUEUOM'], inplace=True)

### culture <== chartevent

In [97]:
culture_list = ['Arterial Line Tip Cultured', 'CCO PAC Line Tip Cultured', 'Cordis/Introducer Line Tip Cultured',
     'Dialysis Catheter Tip Cultured', 'Tunneled (Hickman) Line Tip Cultured', 'IABP Line Tip Cultured',
     'Midline Tip Cultured', 'Multi Lumen Line Tip Cultured', 'PA Catheter Line Tip Cultured',
     'Pheresis Catheter Line Tip Cultured', 'PICC Line Tip Cultured', 'Trauma Line Tip Cultured',
     'Indwelling Port (PortaCath) Line Tip Cultured', 'Presep Catheter Line Tip Cultured', 'AVA Line Tip Cultured',
     'Triple Introducer Line Tip Cultured', 'Sheath Line Tip Cultured', 'ICP Line Tip Cultured']

In [98]:
culture_df = chart_df[ chart_df.LABEL.isin(culture_list)]
chart_df   = chart_df[~chart_df.LABEL.isin(culture_list)]

culture_df['LABEL'] = 'Blood Culture'

In [99]:
chart_df.head(3)

In [100]:
print(chart_df.SUBJECT_ID.nunique())
print(chart_df.HADM_ID.nunique())
print(chart_df.ICUSTAY_ID.nunique())

15099
17928
19187


In [101]:
culture_df.head(3)

In [102]:
print(culture_df.SUBJECT_ID.nunique())
print(culture_df.HADM_ID.nunique())
print(culture_df.ICUSTAY_ID.nunique())

2721
2837
2911


### datetiemevent

In [103]:
dateevents = []

for chunk in pd.read_csv(mimiciii + "DATETIMEEVENTS.csv", chunksize=10000):
    chunk = chunk[chunk.ICUSTAY_ID.isin(cohort_stay_id_icu)]
    chunk = chunk[chunk.ERROR != 1]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME', 'VALUE']]

    if chunk.shape[0] > 0:
        dateevents.append(chunk)
        
datetimeevents = pd.concat(dateevents)

In [104]:
datetimeevents = pd.merge(datetimeevents, d_icuitems, on=['ITEMID'], how='left')
datetimeevents = datetimeevents[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'VALUE', 'LABEL']]
datetimeevents['VALUE'] = pd.to_datetime(datetimeevents['VALUE'], errors='coerce')
datetimeevents = datetimeevents[datetimeevents.VALUE.notnull()]
datetimeevents.rename(columns={"VALUE": "CHARTTIME"}, inplace=True)
datetimeevents['VALUE'] = 1

In [105]:
date_var = ['22 Gauge Insertion Date', '20 Gauge Insertion Date', '18 Gauge Insertion Date',
            'Arterial line Insertion Date', 'Arterial line Tubing Change', 'Arterial Line Dressing Change',
            'Multi Lumen Insertion Date', 'Multi Lumen Cap Change', 'Multi Lumen Dressing Change',
            'Multi Lumen Tubing Change']

datetimeevents = datetimeevents[datetimeevents.LABEL.isin(date_var)]

In [106]:
datetimeevents.head(3)

In [107]:
print(datetimeevents.SUBJECT_ID.nunique())
print(datetimeevents.HADM_ID.nunique())
print(datetimeevents.ICUSTAY_ID.nunique())
print(datetimeevents.LABEL.nunique())

14374
16958
17997
10


### outputevent

In [108]:
outevents = []

for chunk in pd.read_csv(mimiciii + "OUTPUTEVENTS.csv", chunksize=10000):
    chunk = chunk[chunk.ICUSTAY_ID.isin(cohort_stay_id_icu)]

    if chunk.shape[0] > 0:
        outevents.append(chunk)
        
outputevents = pd.concat(outevents)

In [109]:
outputevents = pd.merge(outputevents, d_icuitems, on=['ITEMID'], how='left')
outputevents['CHARTTIME'] = pd.to_datetime(outputevents['CHARTTIME'])
outputevents = outputevents[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE']]

In [110]:
Urine_output = ["Urine Out Foley", "Urine .", "Urine Out Void", "Urine Out Condom Cath", "Urine Out Suprapubic",
                "Urine Out IleoConduit", "Urine Out Incontinent", "Urine Out Rt Nephrostomy", "Urine Out Other",
                "Urine Out Lt Nephrostomy", "Urine Out Straight Cath", "Urine Out Incontinent", "Ileoconduit",
                "Urine Out Ureteral Stent #1", "Urine Out Ureteral Stent #2", "Foley", "Void", "Condom Cath",
                "Suprapubic", "R Nephrostomy", "L Nephrostomy", "Straight Cath", "R Ureteral Stent",
                "L Ureteral Stent", "GU Irrigant Volume In", "GU Irrigant/Urine Volume Out", "OR Out PACU Urine",
                "OR Out OR Urine", "OR Urine"]

Stool = ['Stool Out Stool', 'Stool', 'Stool Out Ileostomy', 'Stool Out Fecal Bag', 'Stool Out Colostomy',
         'Stool Out Rectal Tube', 'Stool Out (non-specific)', 'Stool .']

In [111]:
outputevents = outputevents[outputevents.LABEL.isin(Urine_output) | outputevents.LABEL.isin(Stool)]

In [112]:
outputevents.loc[outputevents.LABEL.isin(Urine_output), 'LABEL'] = 'UrineOutput_IO'
outputevents.loc[outputevents.LABEL.isin(Stool),        'LABEL'] = 'Stool_IO'

In [113]:
outputevents.head(3)

In [114]:
print(outputevents.SUBJECT_ID.nunique())
print(outputevents.HADM_ID.nunique())
print(outputevents.ICUSTAY_ID.nunique())
print(outputevents.LABEL.nunique())

14845
17535
18692
2


### inputevent

In [115]:
inputevent_mv = []

for chunk in pd.read_csv(mimiciii + "INPUTEVENTS_MV.csv", chunksize=10000):
    chunk = chunk[chunk.ICUSTAY_ID.isin(cohort_stay_id_icu)]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'STARTTIME', 'ITEMID', 'AMOUNT', 'AMOUNTUOM']]

    if chunk.shape[0] > 0:
        inputevent_mv.append(chunk)
        
inputevent_mv = pd.concat(inputevent_mv)
inputevent_mv.rename(columns={"STARTTIME": "CHARTTIME"}, inplace=True)

In [116]:
inputevent_cv = []

for chunk in pd.read_csv(mimiciii + "INPUTEVENTS_CV.csv", chunksize=10000):
    chunk = chunk[chunk.ICUSTAY_ID.isin(cohort_stay_id_icu)]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'ITEMID', 'AMOUNT', 'AMOUNTUOM']]

    if chunk.shape[0] > 0:
        inputevent_cv.append(chunk)
        
inputevent_cv = pd.concat(inputevent_cv)

In [117]:
inputevents = pd.concat([inputevent_cv, inputevent_mv])

In [118]:
inputevents = pd.merge(inputevents, d_icuitems, on=['ITEMID'], how='left')
inputevents['CHARTTIME'] = pd.to_datetime(inputevents['CHARTTIME'])

In [119]:
inputevents.loc[(inputevents.LABEL == 'Ceftriaxone')   & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Cefazolin')     & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Cefepime')      & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Ceftazidime')   & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Vancomycin')    & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Clindamycin')   & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Metronidazole') & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Meropenem')     & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Tobramycin')    & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Acyclovir')     & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Linezolid')     & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Azithromycin')  & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Voriconazole')  & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Levofloxacin')  & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Micafungin')    & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Fluconazole')   & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Ranitidine (Prophylaxis)') & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1
inputevents.loc[(inputevents.LABEL == 'Famotidine (Pepcid)')      & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[(inputevents.LABEL == 'Pantoprazole (Protonix)')  & (inputevents.AMOUNT.notnull()), 'AMOUNT'] = 1 
inputevents.loc[((inputevents.LABEL == 'Propofol')          & (inputevents.AMOUNTUOM == 'mcg')) , 'AMOUNT'] = np.nan 
inputevents.loc[((inputevents.LABEL == 'Ciprofloxacin')     & (inputevents.AMOUNTUOM == 'mg'))  , 'AMOUNT'] = np.nan 
inputevents.loc[((inputevents.LABEL == 'Fentanyl')          & (inputevents.AMOUNTUOM == 'mcg')) , 'AMOUNT'] = inputevents['AMOUNT']/1000
inputevents.loc[((inputevents.LABEL == 'Digoxin (Lanoxin)') & (inputevents.AMOUNTUOM == 'mcg')) , 'AMOUNT'] = inputevents['AMOUNT']/1000
inputevents.loc[((inputevents.LABEL == 'Thiamine')          & (inputevents.AMOUNTUOM == 'dose')), 'AMOUNT'] = inputevents['AMOUNT']*1000 
inputevents.loc[((inputevents.LABEL == 'Beneprotein')       & (inputevents.AMOUNTUOM == 'dose')), 'AMOUNT'] = inputevents['AMOUNT']*100 
inputevents.loc[((inputevents.LABEL == 'Digoxin (Lanoxin)') & (inputevents.AMOUNTUOM == 'mg'))  , 'AMOUNT'] = inputevents['AMOUNT']*1000 
inputevents.loc[((inputevents.LABEL == 'Fentanyl (Concentrate)')       & (inputevents.AMOUNTUOM == 'mcg')), 'AMOUNT'] = inputevents['AMOUNT']/1000
inputevents.loc[((inputevents.LABEL == 'Pre-Admission/Non-ICU Intake') & (inputevents.AMOUNTUOM == 'L'))  , 'AMOUNT'] = inputevents['AMOUNT']*1000

In [120]:
Dobutamine_ID = [221653]
Milrinone_ID  = [221986]
NeuroblockAgent_ID = [222062, 221555]
KIV_ID = [225166, 225834, 222139, 225925]
CaIV_ID = [228317, 229640]
CanonIV_ID = [221456, 227525, 229618]
MgIV_ID = [227524]
MgnonIV_ID = [227523, 222011]
PIV_ID = [225925, 225834]
PnonIV_ID = [225890]
Fluids_ID = [221212, 221213, 225161, 226401]
BetaBlockers_ID = [225974, 221429]
CaBlockers_ID = [221347, 229654, 228339, 221468]
LoopDiuretics_ID = [221794, 228340]
TPNutrition_ID = [225916, 225917]
Fentanyl_ID = [225972, 225942, 221744]
Albumin_ID = [220861, 220862, 220863, 220864]
Insulin_ID = [223257, 223258, 223259, 223260, 223261, 223262, 229299, 229619]
Dextrose_ID = [220949, 220950, 220951, 220952, 220963, 220964, 220965, 220966, 220967, 220968,
               221000, 221002, 221014, 221017, 228140, 228141, 228142] 
PNutrition_ID = [227090, 222190, 225801, 225916, 225917, 225920, 225947, 225948, 225969]
POnutrition_ID = [221036, 225931, 226880, 227518, 226017, 226016, 226018, 226019, 226028, 226029, 226030, 226881]
Vasopressors_ID = [222315, 221289, 221906, 229617, 221749, 229630, 221662, 229632, 229631]    

In [121]:
Propofol = ['Propofol']

Fentanyl = ['Fentanyl', 'Fentanyl (Concentrate)', 'Fentanyl Drip', 'Fentanyl (Conc)', 'Fentanyl Base',
            'Fentanyl bolus:']

Insulin = ['Insulin', 'Insulin - Regular', 'Insulin - Humalog', 'Insulin - Glargine', 'Regular Insulin',
           'Insulin - NPH', 'Insulin - 70/30', 'Insulin Drip', 'Insulin - Humalog 75/25', 'insulin carrier',
           'Insulin Carrier', 'NS Insulin Carrier', 'PORK INSULIN GTT', 'Insulin carrier']

Heparin = ['Heparin', 'Heparin Sodium (Prophylaxis)', 'Heparin Sodium', '.45NS + .5:1 Heparin',
           '.45NS + 1:1 Heparin', '.9NS + 1:1 Heparin', 'Heparin(10 units/cc)', '.9NS + 0.5:1 heparin',
           '.25 NS+0.5:1 Heparin', 'Na Acetate/Heparin', 'NA Acetate w/Heparin', 'PN D9.5 w/ heparin',
           '.25 NS +1:1 Heparin', 'heparin via sheaths', 'D10W with heparin', 'CRRT HEPARIN',
           '1000NS/1000UHEPARIN', 'TPND9.5+Heparin']

Midazolam = ['Midazolam', 'Midazolam (Versed)', 'Midazolam(Versed)']

Dexmedetomidine = ['Dexmedetomidine (Precedex)', 'Dexmedetomide', 'dexmedetomidine', 'DEXMEDETOMIDINE']

Vassopressin = ['vassopressin']

Albumin = ['Albumin 25%', 'Albumin 5%', 'albumin 12.5%', 'Serum Albumin 5%', 'albumin', 'Albumin', 'ALBUMIN',
           'Albumin (human) 25%', 'Albumin 12.5%', '25% Albumin', 'OR Albumin']

Ceftriaxone = ['Ceftriaxone']

Cefazolin = ['Cefazolin']

Cefepime = ['Cefepime']

Ceftazidime = ['Ceftazidime']

Vancomycin = ['Vancomycin', 'Vancomycin enema']

Clindamycin = ['Clindamycin']

Metronidazole = ['Metronidazole']

Meropenem = ['Meropenem']

Acyclovir = ['Acyclovir']

Azithromycin = ['Azithromycin']

Levofloxacin = ['Levofloxacin']

Micafungin = ['Micafungin']

Fluconazole = ['Fluconazole', 'FLUCONAZOLE']

SodiumChloride = ['Sodium Chloride']

Thiamine = ['Thiamine']

Dobutamine = ['Dobutamine', 'Dobutamine Drip']

Milrinone = ['Milrinone']

Fluids = ['PO Fluids', 'angio fluids /anes', 'EW fluids', 'ER FLUIDS', 'CC7 FLUIDS', 'iv fluids on floor.',
          'pharesis fluids']

OralIntake = ['oral', 'Oral']

PO = ['Po Intake', 'PO Intake']

IVPB = ['IVPB', 'kcl 20 meq ivpb']

Crystalloids = ['OR Crystalloid', 'OR Crystalloid Intake', 'PACU Crystalloids', 'PACU Crystalloid Intake',
                'ER CRYSTALLOIDS', 'er crystalloid', 'er crystalloids', 'angio crystalloid', 'ER CRYSTALLOID',
                'crystalloids in ango', 'angio crystalloids', 'ED crystalloid', 'EW crystalloids',
                'ECMO CRYSTALLOID', 'ANGIO CRYSTALLOID', 'CRYSTALLOID IN ANGIO', 'OR INPUT CRYSTALLOID',
                'E.R. CRYSTALLOID', 'crystalloid farr 5', 'ed crystalloids', 'CC7 CRYSTALLOIDS',
                'crystalloid- ed', 'floor crystalloid', 'ANGIO CRYSTALLOIDS', 'crystalloid from f10',
                'farr 7 crystalloid', 'ED CRYSTALLOID', 'IR CRYSTALLOID', 'ed crystalloid', 'FLOOR CRYSTALLOIDS',
                'Floor crystalloid']

NSIVF = ['osh ivf of ns', 'ns ivfin ed']

Norepinephrine = ['Norepinephrine']

Amiodarone = ['Amiodarone', 'Amiodarone 600/500', 'Amiodarone mg/hr']

Phenylephrine = ['Phenylephrine']

Epinephrine = ['Epinephrine-k', 'Epinephrine', 'Epinephrine Drip']

Nicardipine = ['Nicardipine', 'Nicardipine (mg/h)', 'nicardipine gtt', 'nicardipine HCL', 'Nicardipine gtt',
               'Nicardipine mg/hr', 'nicardipine drip']

Pantoprazole = ['Pantoprazole (Protonix)', 'pantoprazole', 'Pantoprazole', 'PANTOPRAZOLE', 'pantoprazole gtt',
                'pantoprazole 8mg/hr', 'PANTOPRAZOLE GTT', 'pantoprazole mg/hr', 'Pantoprazole 8mg/hr',
                'Pantoprazole gtt']

Diltiazem = ['Diltiazem']

Nitroglycerin = ['Nitroglycerine-k', 'Nitroglycerin', 'Nitroglycerine']

In [122]:
inputevents = inputevents[inputevents.ITEMID.isin(Dobutamine_ID)       | inputevents.ITEMID.isin(Milrinone_ID)    |
                          inputevents.ITEMID.isin(NeuroblockAgent_ID)  | inputevents.ITEMID.isin(KIV_ID)          |
                          inputevents.ITEMID.isin(CaIV_ID)             | inputevents.ITEMID.isin(CanonIV_ID)      |
                          inputevents.ITEMID.isin(MgIV_ID)             | inputevents.ITEMID.isin(MgnonIV_ID)      |
                          inputevents.ITEMID.isin(PIV_ID)              | inputevents.ITEMID.isin(PnonIV_ID)       |
                          inputevents.ITEMID.isin(Fluids_ID)           | inputevents.ITEMID.isin(BetaBlockers_ID) |
                          inputevents.ITEMID.isin(CaBlockers_ID)       | inputevents.ITEMID.isin(LoopDiuretics_ID)|
                          inputevents.ITEMID.isin(TPNutrition_ID)      | inputevents.ITEMID.isin(Fentanyl_ID)     |
                          inputevents.ITEMID.isin(Albumin_ID)          | inputevents.ITEMID.isin(Insulin_ID)      |
                          inputevents.ITEMID.isin(Dextrose_ID)         | inputevents.ITEMID.isin(PNutrition_ID)   |
                          inputevents.ITEMID.isin(POnutrition_ID)      | inputevents.ITEMID.isin(Vasopressors_ID) |

                          inputevents.LABEL.isin(Propofol)             | inputevents.LABEL.isin(Fentanyl)         |
                          inputevents.LABEL.isin(Insulin)              | inputevents.LABEL.isin(Heparin)          |
                          inputevents.LABEL.isin(Midazolam)            | inputevents.LABEL.isin(Dexmedetomidine)  |
                          inputevents.LABEL.isin(Vassopressin)         | inputevents.LABEL.isin(Albumin)          |
                          inputevents.LABEL.isin(Ceftriaxone)          | inputevents.LABEL.isin(Cefazolin)        |
                          inputevents.LABEL.isin(Cefepime)             | inputevents.LABEL.isin(Ceftazidime)      |
                          inputevents.LABEL.isin(Vancomycin)           | inputevents.LABEL.isin(Clindamycin)      |
                          inputevents.LABEL.isin(Metronidazole)        | inputevents.LABEL.isin(Meropenem)        |
                          inputevents.LABEL.isin(Acyclovir)            | inputevents.LABEL.isin(Azithromycin)     |
                          inputevents.LABEL.isin(Levofloxacin)         | inputevents.LABEL.isin(Micafungin)       |
                          inputevents.LABEL.isin(Fluconazole)          | inputevents.LABEL.isin(Thiamine)         |
                          inputevents.LABEL.isin(Dobutamine)           | inputevents.LABEL.isin(Milrinone)        |
                          inputevents.LABEL.isin(Fluids)               | inputevents.LABEL.isin(OralIntake)       |
                          inputevents.LABEL.isin(PO)                   | inputevents.LABEL.isin(SodiumChloride)   |
                          inputevents.LABEL.isin(IVPB)                 | inputevents.LABEL.isin(Stool)            |
                          inputevents.LABEL.isin(Crystalloids)         | inputevents.LABEL.isin(NSIVF)            |
                          inputevents.LABEL.isin(Norepinephrine)       | inputevents.LABEL.isin(Amiodarone)       |
                          inputevents.LABEL.isin(Phenylephrine)        | inputevents.LABEL.isin(Epinephrine)      |
                          inputevents.LABEL.isin(Nicardipine)          | inputevents.LABEL.isin(Pantoprazole)     |
                          inputevents.LABEL.isin(Diltiazem)            | inputevents.LABEL.isin(Nitroglycerin) ] 

In [123]:
inputevents.loc[inputevents.ITEMID.isin(Dobutamine_ID),      'LABEL'] = 'Dobutamine_IO'
inputevents.loc[inputevents.ITEMID.isin(Milrinone_ID),       'LABEL'] = 'Milrinone_IO'
inputevents.loc[inputevents.ITEMID.isin(NeuroblockAgent_ID), 'LABEL'] = 'NeuroblockAgent_IO'
inputevents.loc[inputevents.ITEMID.isin(KIV_ID),             'LABEL'] = 'K-IV_IO'
inputevents.loc[inputevents.ITEMID.isin(CaIV_ID),            'LABEL'] = 'Ca-IV_IO'
inputevents.loc[inputevents.ITEMID.isin(CanonIV_ID),         'LABEL'] = 'Ca-nonIV_IO'
inputevents.loc[inputevents.ITEMID.isin(MgIV_ID),            'LABEL'] = 'Mg-IV_IO'
inputevents.loc[inputevents.ITEMID.isin(MgnonIV_ID),         'LABEL'] = 'Mg-nonIV_IO'
inputevents.loc[inputevents.ITEMID.isin(PIV_ID),             'LABEL'] = 'P-IV_IO'
inputevents.loc[inputevents.ITEMID.isin(PnonIV_ID),          'LABEL'] = 'P-nonIV_IO'
inputevents.loc[inputevents.ITEMID.isin(Fluids_ID),          'LABEL'] = 'Fluids_IO'
inputevents.loc[inputevents.ITEMID.isin(BetaBlockers_ID),    'LABEL'] = 'BetaBlockers_IO'
inputevents.loc[inputevents.ITEMID.isin(CaBlockers_ID),      'LABEL'] = 'CaBlockers_IO'
inputevents.loc[inputevents.ITEMID.isin(LoopDiuretics_ID),   'LABEL'] = 'LoopDiuretics_IO'
inputevents.loc[inputevents.ITEMID.isin(TPNutrition_ID),     'LABEL'] = 'TPNutrition_IO'
inputevents.loc[inputevents.ITEMID.isin(Fentanyl_ID),        'LABEL'] = 'Fentanyl_IO'
inputevents.loc[inputevents.ITEMID.isin(Albumin_ID),         'LABEL'] = 'Albumin_IO'
inputevents.loc[inputevents.ITEMID.isin(Insulin_ID),         'LABEL'] = 'Insulin_IO'
inputevents.loc[inputevents.ITEMID.isin(Dextrose_ID),        'LABEL'] = 'Dextrose_IO'
inputevents.loc[inputevents.ITEMID.isin(PNutrition_ID),      'LABEL'] = 'PNutrition_IO'
inputevents.loc[inputevents.ITEMID.isin(POnutrition_ID),     'LABEL'] = 'POnutrition_IO'
inputevents.loc[inputevents.ITEMID.isin(Vasopressors_ID),    'LABEL'] = 'Vasopressors_IO'

In [124]:
inputevents.loc[inputevents.LABEL.isin(Propofol)  ,       'LABEL'] = 'Propofol_IO'
inputevents.loc[inputevents.LABEL.isin(Fentanyl)  ,       'LABEL'] = 'Fentanyl_IO'
inputevents.loc[inputevents.LABEL.isin(Insulin)  ,        'LABEL'] = 'Insulin_IO'
inputevents.loc[inputevents.LABEL.isin(Heparin)  ,        'LABEL'] = 'Heparin_IO'
inputevents.loc[inputevents.LABEL.isin(Midazolam)  ,      'LABEL'] = 'Midazolam_IO'
inputevents.loc[inputevents.LABEL.isin(Dexmedetomidine) , 'LABEL'] = 'Dexmedetomidine_IO'
inputevents.loc[inputevents.LABEL.isin(Vassopressin)  ,   'LABEL'] = 'Vassopressin_IO'
inputevents.loc[inputevents.LABEL.isin(Albumin)  ,        'LABEL'] = 'Albumin_IO'
inputevents.loc[inputevents.LABEL.isin(Ceftriaxone)  ,    'LABEL'] = 'Ceftriaxone_IO'
inputevents.loc[inputevents.LABEL.isin(Cefazolin)  ,      'LABEL'] = 'Cefazolin_IO'
inputevents.loc[inputevents.LABEL.isin(Cefepime)  ,       'LABEL'] = 'Cefepime_IO'
inputevents.loc[inputevents.LABEL.isin(Ceftazidime)  ,    'LABEL'] = 'Ceftazidime_IO'
inputevents.loc[inputevents.LABEL.isin(Vancomycin)  ,     'LABEL'] = 'Vancomycin_IO'
inputevents.loc[inputevents.LABEL.isin(Clindamycin)  ,    'LABEL'] = 'Clindamycin_IO'
inputevents.loc[inputevents.LABEL.isin(Metronidazole)  ,  'LABEL'] = 'Metronidazole_IO'
inputevents.loc[inputevents.LABEL.isin(Meropenem)  ,      'LABEL'] = 'Meropenem_IO'
inputevents.loc[inputevents.LABEL.isin(Acyclovir)  ,      'LABEL'] = 'Acyclovir_IO'
inputevents.loc[inputevents.LABEL.isin(Azithromycin)  ,   'LABEL'] = 'Azithromycin_IO'
inputevents.loc[inputevents.LABEL.isin(Levofloxacin)  ,   'LABEL'] = 'Levofloxacin_IO'
inputevents.loc[inputevents.LABEL.isin(Micafungin)  ,     'LABEL'] = 'Micafungin_IO'
inputevents.loc[inputevents.LABEL.isin(Fluconazole)  ,    'LABEL'] = 'Fluconazole_IO'
inputevents.loc[inputevents.LABEL.isin(Thiamine)  ,       'LABEL'] = 'Thiamine_IO'
inputevents.loc[inputevents.LABEL.isin(Dobutamine)  ,     'LABEL'] = 'Dobutamine_IO'
inputevents.loc[inputevents.LABEL.isin(Milrinone)  ,      'LABEL'] = 'Milrinone_IO'
inputevents.loc[inputevents.LABEL.isin(Fluids)  ,         'LABEL'] = 'Fluids_IO'
inputevents.loc[inputevents.LABEL.isin(OralIntake)  ,     'LABEL'] = 'OralIntake_IO'
inputevents.loc[inputevents.LABEL.isin(PO)  ,             'LABEL'] = 'P.O._IO'
inputevents.loc[inputevents.LABEL.isin(SodiumChloride)  , 'LABEL'] = 'SodiumChloride_IO'
inputevents.loc[inputevents.LABEL.isin(IVPB)  ,           'LABEL'] = 'IVPB_IO'
inputevents.loc[inputevents.LABEL.isin(Stool)  ,          'LABEL'] = 'Stool_IO'
inputevents.loc[inputevents.LABEL.isin(Crystalloids)  ,   'LABEL'] = 'Crystalloids_IO'
inputevents.loc[inputevents.LABEL.isin(NSIVF)  ,          'LABEL'] = 'NSIVF_IO'
inputevents.loc[inputevents.LABEL.isin(Norepinephrine)  , 'LABEL'] = 'Norepinephrine_IO'
inputevents.loc[inputevents.LABEL.isin(Amiodarone)  ,     'LABEL'] = 'Amiodarone_IO'
inputevents.loc[inputevents.LABEL.isin(Phenylephrine)  ,  'LABEL'] = 'Phenylephrine_IO'
inputevents.loc[inputevents.LABEL.isin(Epinephrine)  ,    'LABEL'] = 'Epinephrine_IO'
inputevents.loc[inputevents.LABEL.isin(Nicardipine)  ,    'LABEL'] = 'Nicardipine_IO'
inputevents.loc[inputevents.LABEL.isin(Pantoprazole)  ,   'LABEL'] = 'Pantoprazole_IO'
inputevents.loc[inputevents.LABEL.isin(Diltiazem)  ,      'LABEL'] = 'Diltiazem_IO'
inputevents.loc[inputevents.LABEL.isin(Nitroglycerin)  ,  'LABEL'] = 'Nitroglycerin_IO'

In [125]:
inputevents.drop(columns=['ITEMID', 'AMOUNTUOM'], inplace=True)
inputevents = inputevents[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'AMOUNT']]
inputevents.columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE']

In [126]:
inputevents.head()

In [127]:
print(inputevents.SUBJECT_ID.nunique())
print(inputevents.HADM_ID.nunique())
print(inputevents.ICUSTAY_ID.nunique())
print(inputevents.LABEL.nunique())

14832
17591
18787
47


### Noteevents

In [128]:
note = []
discharge_note = []

for chunk in pd.read_csv(mimiciii + "NOTEEVENTS.csv", chunksize=10000): 
    chunk = chunk[chunk.SUBJECT_ID.isin(cohort_subject_id_icu)]
    chunk = chunk[['SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CHARTTIME', 'CATEGORY', 'ROW_ID']]
    note_chunk = chunk[chunk.CATEGORY != 'Discharge summary']
    discharge_chunk = chunk[chunk.CATEGORY == 'Discharge summary']

    if note_chunk.shape[0] > 0:
        note.append(note_chunk)
        
    if discharge_chunk.shape[0] > 0:
        discharge_note.append(discharge_chunk)
        
note = pd.concat(note)
discharge_note = pd.concat(discharge_note)

In [129]:
note['CHARTDATE'] = pd.to_datetime(note['CHARTDATE'])
note['CHARTTIME'] = pd.to_datetime(note['CHARTTIME'])
note.loc[note['CHARTTIME'].isnull(),  'CHARTTIME'] = note['CHARTDATE']

discharge_note['CHARTDATE'] = pd.to_datetime(discharge_note['CHARTDATE'])
discharge_note['CHARTTIME'] = pd.to_datetime(discharge_note['CHARTTIME'])
discharge_note.loc[discharge_note['CHARTTIME'].isnull(),  'CHARTTIME'] = discharge_note['CHARTDATE']

note.drop(columns=['CHARTDATE'], inplace=True)
discharge_note.drop(columns=['CHARTDATE'], inplace=True)

<ipython-input>:7: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  discharge_note.loc[discharge_note['CHARTTIME'].isnull(),  'CHARTTIME'] = discharge_note['CHARTDATE']


In [130]:
note = pd.merge(note, cohort_icu_df, on=['SUBJECT_ID'], how='left')
note['CHARTTIME'] = pd.to_datetime(note['CHARTTIME'])
note = note[(note['CHARTTIME'] >= note['INTIME']) & (note['CHARTTIME'] <= note['OUTTIME'])]

discharge_note = pd.merge(discharge_note, cohort_icu_df, on=['HADM_ID'], how='left')
discharge_note['CHARTTIME'] = pd.to_datetime(discharge_note['CHARTTIME'])
discharge_note = discharge_note[(discharge_note['CHARTTIME'] >= discharge_note['INTIME'])]

In [131]:
note = note[['SUBJECT_ID', 'HADM_ID_y', 'ICUSTAY_ID', 'CHARTTIME', 'ROW_ID']]
note.rename(columns={"HADM_ID_y": "HADM_ID"}, inplace=True)

note['LABEL'] = 'Note'
note = note[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'ROW_ID']]
note.columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE']

In [132]:
discharge_note = discharge_note[['SUBJECT_ID_y', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'ROW_ID']]
discharge_note.rename(columns={"SUBJECT_ID_y": "SUBJECT_ID"}, inplace=True)

discharge_note['LABEL'] = 'Discharge_Note'
discharge_note = discharge_note[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'ROW_ID']]
discharge_note.columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LABEL', 'VALUE']

In [133]:
note.head(2)

In [134]:
discharge_note.head(2)

In [135]:
print(note.SUBJECT_ID.nunique())
print(note.HADM_ID.nunique())
print(note.ICUSTAY_ID.nunique())

13639
16032
17073


In [136]:
print(discharge_note.SUBJECT_ID.nunique())
print(discharge_note.HADM_ID.nunique())
print(discharge_note.ICUSTAY_ID.nunique())

14446
17318
18562


### Save

In [137]:
cohort_df.to_csv(output       + 'demographic.csv',    index=False)
prescriptions.to_csv(output   + 'prescriptions.csv',  index=False)
lab_df.to_csv(output          + 'lab_event.csv',      index=False)
chart_df.to_csv(output        + 'chart_event.csv',    index=False)
culture_df.to_csv(output      + 'culture_event.csv',  index=False)
microbio_df.to_csv(output     + 'microbio_event.csv', index=False)
datetimeevents.to_csv(output  + 'datetimeevents.csv', index=False)
outputevents.to_csv(output    + 'outputevents.csv'  , index=False)
inputevents.to_csv(output     + 'inputevents.csv',    index=False)
note.to_csv(output            + 'note.csv',           index=False)
discharge_note.to_csv(output  + 'discharge_note.csv', index=False)

In [17]:
with open(output + 'race_dictionary.pkl', 'wb') as f:
    pickle.dump(race_dictionary, f)